In [ ]:
import sys
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Python:", sys.executable)
print("Rasterio:", rasterio.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

print("\nEnvironment is ready.")

In [ ]:
# ============================================================
# Cell 2: Define the 14 frozen Paper 1 input rasters
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path(r"C:\project 3\Paper1_SSL")
HARMONIZED_DIR = PROJECT_ROOT / "02_data" / "harmonized_4km"
MASK_DIR = PROJECT_ROOT / "02_data" / "masks"

raster_files = {
    "RTE_TMI": HARMONIZED_DIR / "TMI_RTE_4km.tif",

    "CBG": HARMONIZED_DIR / "P1_CBG_4km.tif",
    "CBG_RES": HARMONIZED_DIR / "P1_CBG_Res_4km.tif",
    "HGM": HARMONIZED_DIR / "P1_HGM_4km_FINAL.tif",

    "K": HARMONIZED_DIR / "P1_K_4km.tif",
    "eTh": HARMONIZED_DIR / "P1_Th_4km.tif",
    "eU": HARMONIZED_DIR / "P1_U_4km.tif",

    "CPD": HARMONIZED_DIR / "P1_CPD_4km.tif",

    "MAG_LD": HARMONIZED_DIR / "P1_MAG_LD_4km.tif",
    "GRAV_LD": HARMONIZED_DIR / "P1_GRAV_LD_4km.tif",
    "DEM_LD": HARMONIZED_DIR / "P1_DEM_LD_4km.tif",
    "ID": HARMONIZED_DIR / "P1_ID_4km.tif",

    "DEM": HARMONIZED_DIR / "P1_DEM_4km.tif",
    "SLOPE": HARMONIZED_DIR / "P1_SLOPE_4km.tif",
}

master_raster = MASK_DIR / "Paper1_MASTER_4km.tif"

print("Number of frozen input channels:", len(raster_files))
print("\nRaster inventory:")

for name, path in raster_files.items():
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{name:10s} : {status:7s} : {path.name}")

print("\nMaster raster:")
print("FOUND" if master_raster.exists() else "MISSING", ":", master_raster.name)

In [ ]:
# ============================================================
# Cell 3: Verify raster geometry, CRS, transform, and NoData
# ============================================================

records = []

# Read master raster metadata first
with rasterio.open(master_raster) as src:
    master_meta = {
        "width": src.width,
        "height": src.height,
        "crs": src.crs,
        "transform": src.transform,
        "bounds": src.bounds,
        "res": src.res,
        "nodata": src.nodata,
    }

print("MASTER RASTER")
print("-" * 60)
print("Shape      :", (master_meta["height"], master_meta["width"]))
print("CRS        :", master_meta["crs"])
print("Resolution :", master_meta["res"])
print("Bounds     :", master_meta["bounds"])
print("NoData     :", master_meta["nodata"])

print("\nINPUT RASTER CHECK")
print("-" * 60)

for name, path in raster_files.items():

    with rasterio.open(path) as src:

        same_shape = (
            src.width == master_meta["width"]
            and src.height == master_meta["height"]
        )

        same_crs = src.crs == master_meta["crs"]

        same_transform = src.transform.almost_equals(
            master_meta["transform"]
        )

        same_bounds = np.allclose(
            [src.bounds.left,
             src.bounds.bottom,
             src.bounds.right,
             src.bounds.top],
            [master_meta["bounds"].left,
             master_meta["bounds"].bottom,
             master_meta["bounds"].right,
             master_meta["bounds"].top],
            atol=1e-6
        )

        same_res = np.allclose(
            src.res,
            master_meta["res"],
            atol=1e-6
        )

        passed = (
            same_shape
            and same_crs
            and same_transform
            and same_bounds
            and same_res
        )

        records.append({
            "channel": name,
            "rows": src.height,
            "cols": src.width,
            "xres": src.res[0],
            "yres": src.res[1],
            "crs": str(src.crs),
            "nodata": src.nodata,
            "same_shape": same_shape,
            "same_crs": same_crs,
            "same_transform": same_transform,
            "same_bounds": same_bounds,
            "same_res": same_res,
            "PASS": passed,
        })

        print(
            f"{name:10s} "
            f"Shape={same_shape}  "
            f"CRS={same_crs}  "
            f"Transform={same_transform}  "
            f"Bounds={same_bounds}  "
            f"Res={same_res}  "
            f"=> {'PASS' if passed else 'FAIL'}"
        )

# Convert results to a dataframe
geometry_qc = pd.DataFrame(records)

print("\nSummary")
print("-" * 60)
print("Passed:", geometry_qc["PASS"].sum(), "/", len(geometry_qc))

if geometry_qc["PASS"].all():
    print("\nAll 14 rasters are geometrically aligned with the master grid.")
else:
    print("\nWARNING: At least one raster failed the geometry check.")

In [ ]:
# ============================================================
# Cell 4: Read rasters, convert NoData to NaN,
#         build the 14-channel cube and common valid-data mask
# ============================================================

raster_arrays = {}
valid_masks = {}

for name, path in raster_files.items():

    with rasterio.open(path) as src:

        # Read first raster band as float64
        arr = src.read(1).astype("float64")

        # ----------------------------------------------------
        # Convert declared NoData values to NaN
        # ----------------------------------------------------
        if src.nodata is not None:
            arr[np.isclose(arr, src.nodata)] = np.nan

        # Also protect against non-finite values
        arr[~np.isfinite(arr)] = np.nan

        raster_arrays[name] = arr

        # True where the channel has a valid numerical value
        valid_masks[name] = np.isfinite(arr)

# ------------------------------------------------------------
# Stack arrays into one raster cube
# Shape will be: (rows, cols, channels)
# ------------------------------------------------------------

channel_names = list(raster_files.keys())

cube = np.stack(
    [raster_arrays[name] for name in channel_names],
    axis=-1
)

# ------------------------------------------------------------
# Common valid-data mask:
# a cell is valid only if ALL 14 channels are valid there
# ------------------------------------------------------------

common_mask = np.all(np.isfinite(cube), axis=-1)

total_cells = common_mask.size
valid_cells = int(common_mask.sum())
invalid_cells = int(total_cells - valid_cells)

valid_pct = 100.0 * valid_cells / total_cells
invalid_pct = 100.0 * invalid_cells / total_cells

print("Cube shape:", cube.shape)
print("Expected  :", (42, 69, 14))

print("\nCOMMON VALID-DATA MASK")
print("-" * 60)
print("Total master-grid cells :", total_cells)
print("Valid across all 14     :", valid_cells)
print("Invalid in >=1 channel  :", invalid_cells)
print(f"Valid percentage        : {valid_pct:.2f}%")
print(f"Invalid percentage      : {invalid_pct:.2f}%")

# ------------------------------------------------------------
# Missing cells by individual channel
# ------------------------------------------------------------

print("\nMISSING CELLS BY CHANNEL")
print("-" * 60)

missing_summary = []

for name in channel_names:

    n_missing = int((~valid_masks[name]).sum())
    pct_missing = 100.0 * n_missing / total_cells

    missing_summary.append({
        "channel": name,
        "missing_cells": n_missing,
        "missing_percent": pct_missing,
    })

    print(
        f"{name:10s} : "
        f"{n_missing:4d} missing "
        f"({pct_missing:6.2f}%)"
    )

missing_df = pd.DataFrame(missing_summary)

print("\nCell 4 completed.")

In [ ]:
# ============================================================
# Cell 5: Visualize and save the common valid-data mask
# ============================================================

from rasterio.transform import array_bounds

# ------------------------------------------------------------
# Read master raster profile
# ------------------------------------------------------------

with rasterio.open(master_raster) as src:
    master_profile = src.profile.copy()
    master_transform = src.transform
    master_crs = src.crs

# ------------------------------------------------------------
# Plot common mask
# ------------------------------------------------------------

plt.figure(figsize=(10, 6))

plt.imshow(
    common_mask,
    cmap="gray",
    origin="upper"
)

plt.title(
    f"Paper 1 Common Valid-Data Mask\n"
    f"{valid_cells} / {total_cells} cells valid ({valid_pct:.2f}%)"
)

plt.xlabel("Column")
plt.ylabel("Row")

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Save mask as GeoTIFF
# 1 = valid in all 14 channels
# 0 = invalid in at least one channel
# ------------------------------------------------------------

mask_output = MASK_DIR / "Paper1_COMMON_MASK_14ch.tif"

mask_profile = master_profile.copy()

mask_profile.update(
    dtype="uint8",
    count=1,
    nodata=None,
    compress="lzw"
)

with rasterio.open(
    mask_output,
    "w",
    **mask_profile
) as dst:

    dst.write(
        common_mask.astype("uint8"),
        1
    )

print("Common mask saved to:")
print(mask_output)

print("\nMask values:")
unique_values, counts = np.unique(
    common_mask.astype("uint8"),
    return_counts=True
)

for value, count in zip(unique_values, counts):
    label = "VALID" if value == 1 else "INVALID"
    print(f"{value} = {label:7s} : {count} cells")

print("\nCell 5 completed.")

In [ ]:
# ============================================================
# Cell 6: Convert valid raster cells to a machine-learning table
#         while preserving row, column, X, and Y coordinates
# ============================================================

# ------------------------------------------------------------
# Get row/column indices of valid cells
# ------------------------------------------------------------

rows, cols = np.where(common_mask)

print("Number of valid cells:", len(rows))

# ------------------------------------------------------------
# Convert raster row/column positions to map coordinates
# using the master raster transform
# ------------------------------------------------------------

xs, ys = rasterio.transform.xy(
    master_transform,
    rows,
    cols,
    offset="center"
)

xs = np.array(xs)
ys = np.array(ys)

# ------------------------------------------------------------
# Extract the 14-channel values only at valid cells
# ------------------------------------------------------------

X_valid = cube[common_mask]

print("X_valid shape:", X_valid.shape)
print("Expected shape :", (valid_cells, 14))

# ------------------------------------------------------------
# Build dataframe
# ------------------------------------------------------------

ml_df = pd.DataFrame(
    X_valid,
    columns=channel_names
)

# Add spatial identifiers at the beginning
ml_df.insert(0, "Y", ys)
ml_df.insert(0, "X", xs)
ml_df.insert(0, "Col", cols)
ml_df.insert(0, "Row", rows)

# Add unique cell identifier
ml_df.insert(
    0,
    "Cell_ID",
    [f"R{r:02d}_C{c:02d}" for r, c in zip(rows, cols)]
)

# ------------------------------------------------------------
# Basic checks
# ------------------------------------------------------------

print("\nDataframe shape:", ml_df.shape)

print("\nFirst 5 rows:")
display(ml_df.head())

print("\nLast 5 rows:")
display(ml_df.tail())

print("\nMissing values in ML table:")
print(ml_df.isna().sum())

# ------------------------------------------------------------
# Save the unscaled analytical table
# ------------------------------------------------------------

TABLE_DIR = PROJECT_ROOT / "03_outputs" / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

table_output = TABLE_DIR / "Paper1_14ch_valid_cells_UNSCALED.csv"

ml_df.to_csv(
    table_output,
    index=False
)

print("\nUnscaled analytical table saved to:")
print(table_output)

print("\nCell 6 completed.")

In [ ]:
# ============================================================
# Cell 7: Descriptive statistics and distribution audit
# ============================================================

from scipy.stats import skew, kurtosis

# Only the 14 geophysical input channels
feature_df = ml_df[channel_names].copy()

summary_records = []

for name in channel_names:

    values = feature_df[name].to_numpy()

    q1 = np.percentile(values, 25)
    median = np.percentile(values, 50)
    q3 = np.percentile(values, 75)

    iqr = q3 - q1

    summary_records.append({
        "channel": name,
        "count": len(values),
        "min": np.min(values),
        "q1": q1,
        "median": median,
        "mean": np.mean(values),
        "q3": q3,
        "max": np.max(values),
        "std": np.std(values, ddof=1),
        "IQR": iqr,
        "skewness": skew(values, bias=False),
        "kurtosis_excess": kurtosis(
            values,
            fisher=True,
            bias=False
        ),
        "zero_count": np.sum(values == 0),
        "zero_percent": 100 * np.mean(values == 0)
    })

summary_df = pd.DataFrame(summary_records)

# ------------------------------------------------------------
# Display summary table
# ------------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

display(
    summary_df.round(4)
)

# ------------------------------------------------------------
# Save descriptive statistics
# ------------------------------------------------------------

stats_output = TABLE_DIR / "Paper1_14ch_descriptive_statistics.csv"

summary_df.to_csv(
    stats_output,
    index=False
)

print("Descriptive statistics saved to:")
print(stats_output)

print("\nCell 7 completed.")

In [ ]:
# ============================================================
# Fix output directory after kernel change
# ============================================================

FIGURE_DIR = PROJECT_ROOT / "03_outputs" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Figure directory:", FIGURE_DIR)

In [ ]:
# ============================================================
# Cell 8: Plot distributions of the 14 frozen input channels
# ============================================================

import math

n_features = len(channel_names)

ncols = 3
nrows = math.ceil(n_features / ncols)

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(15, 18)
)

axes = axes.flatten()

for i, name in enumerate(channel_names):

    values = feature_df[name].to_numpy()

    axes[i].hist(
        values,
        bins=40,
        edgecolor="black",
        linewidth=0.4
    )

    median_value = np.median(values)
    mean_value = np.mean(values)

    axes[i].axvline(
        median_value,
        linestyle="--",
        linewidth=1.2,
        label="Median"
    )

    axes[i].axvline(
        mean_value,
        linestyle=":",
        linewidth=1.2,
        label="Mean"
    )

    skew_value = summary_df.loc[
        summary_df["channel"] == name,
        "skewness"
    ].iloc[0]

    axes[i].set_title(
        f"{name}\nSkewness = {skew_value:.2f}"
    )

    axes[i].set_xlabel("Value")
    axes[i].set_ylabel("Frequency")

# Remove unused subplot axes
for j in range(n_features, len(axes)):
    fig.delaxes(axes[j])

fig.suptitle(
    "Paper 1 — Distribution of the 14 Frozen Input Channels",
    fontsize=16,
    y=1.01
)

plt.tight_layout()

# Save figure
distribution_output = (
    FIGURE_DIR /
    "Paper1_14ch_distribution_histograms.png"
)

plt.savefig(
    distribution_output,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Distribution figure saved to:")
print(distribution_output)

print("\nCell 8 completed.")

In [ ]:
# ============================================================
# Cell 9: Pearson correlation matrix
# ============================================================

# ------------------------------------------------------------
# Calculate Pearson correlations
# ------------------------------------------------------------

pearson_corr = feature_df.corr(method="pearson")

print("Pearson correlation matrix:")
display(pearson_corr.round(3))

# ------------------------------------------------------------
# Plot Pearson correlation matrix
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(12, 10))

im = ax.imshow(
    pearson_corr.values,
    vmin=-1,
    vmax=1,
    aspect="auto"
)

ax.set_xticks(np.arange(len(channel_names)))
ax.set_yticks(np.arange(len(channel_names)))

ax.set_xticklabels(
    channel_names,
    rotation=45,
    ha="right"
)

ax.set_yticklabels(channel_names)

# Add correlation values inside cells
for i in range(len(channel_names)):
    for j in range(len(channel_names)):

        value = pearson_corr.iloc[i, j]

        ax.text(
            j,
            i,
            f"{value:.2f}",
            ha="center",
            va="center",
            fontsize=7
        )

ax.set_title(
    "Paper 1 — Pearson Correlation Matrix\n"
    "14 Frozen Input Channels"
)

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Pearson r")

plt.tight_layout()

# ------------------------------------------------------------
# Save figure and table
# ------------------------------------------------------------

pearson_fig_output = (
    FIGURE_DIR /
    "Paper1_14ch_Pearson_correlation.png"
)

pearson_table_output = (
    TABLE_DIR /
    "Paper1_14ch_Pearson_correlation.csv"
)

plt.savefig(
    pearson_fig_output,
    dpi=300,
    bbox_inches="tight"
)

pearson_corr.to_csv(pearson_table_output)

plt.show()

print("Pearson figure saved to:")
print(pearson_fig_output)

print("\nPearson matrix saved to:")
print(pearson_table_output)

# ------------------------------------------------------------
# List strongest off-diagonal relationships
# ------------------------------------------------------------

pairs = []

for i in range(len(channel_names)):
    for j in range(i + 1, len(channel_names)):

        pairs.append({
            "Variable_1": channel_names[i],
            "Variable_2": channel_names[j],
            "Pearson_r": pearson_corr.iloc[i, j],
            "Abs_r": abs(pearson_corr.iloc[i, j])
        })

pearson_pairs = (
    pd.DataFrame(pairs)
    .sort_values("Abs_r", ascending=False)
    .reset_index(drop=True)
)

print("\nStrongest Pearson relationships:")
display(pearson_pairs.head(15).round(3))

print("\nCell 9 completed.")

In [ ]:
# ============================================================
# Cell 10: Spearman correlation matrix
# ============================================================

# ------------------------------------------------------------
# Calculate Spearman correlations
# ------------------------------------------------------------

spearman_corr = feature_df.corr(method="spearman")

print("Spearman correlation matrix:")
display(spearman_corr.round(3))

# ------------------------------------------------------------
# Plot Spearman correlation matrix
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(12, 10))

im = ax.imshow(
    spearman_corr.values,
    vmin=-1,
    vmax=1,
    aspect="auto"
)

ax.set_xticks(np.arange(len(channel_names)))
ax.set_yticks(np.arange(len(channel_names)))

ax.set_xticklabels(
    channel_names,
    rotation=45,
    ha="right"
)

ax.set_yticklabels(channel_names)

for i in range(len(channel_names)):
    for j in range(len(channel_names)):

        value = spearman_corr.iloc[i, j]

        ax.text(
            j,
            i,
            f"{value:.2f}",
            ha="center",
            va="center",
            fontsize=7
        )

ax.set_title(
    "Paper 1 — Spearman Correlation Matrix\n"
    "14 Frozen Input Channels"
)

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Spearman ρ")

plt.tight_layout()

# ------------------------------------------------------------
# Save figure and table
# ------------------------------------------------------------

spearman_fig_output = (
    FIGURE_DIR /
    "Paper1_14ch_Spearman_correlation.png"
)

spearman_table_output = (
    TABLE_DIR /
    "Paper1_14ch_Spearman_correlation.csv"
)

plt.savefig(
    spearman_fig_output,
    dpi=300,
    bbox_inches="tight"
)

spearman_corr.to_csv(spearman_table_output)

plt.show()

print("Spearman figure saved to:")
print(spearman_fig_output)

print("\nSpearman matrix saved to:")
print(spearman_table_output)

# ------------------------------------------------------------
# Strongest Spearman relationships
# ------------------------------------------------------------

pairs = []

for i in range(len(channel_names)):
    for j in range(i + 1, len(channel_names)):

        pairs.append({
            "Variable_1": channel_names[i],
            "Variable_2": channel_names[j],
            "Spearman_rho": spearman_corr.iloc[i, j],
            "Abs_rho": abs(spearman_corr.iloc[i, j])
        })

spearman_pairs = (
    pd.DataFrame(pairs)
    .sort_values("Abs_rho", ascending=False)
    .reset_index(drop=True)
)

print("\nStrongest Spearman relationships:")
display(spearman_pairs.head(15).round(3))

print("\nCell 10 completed.")

In [ ]:
# ============================================================
# Cell 11: Variance Inflation Factor (VIF) audit
# ============================================================

from statsmodels.stats.outliers_influence import variance_inflation_factor

# ------------------------------------------------------------
# Prepare feature matrix
# ------------------------------------------------------------

X_vif = feature_df.copy()

# VIF is based on linear dependence among predictors.
# Standardization is not required mathematically for VIF,
# but centering/scaling improves numerical stability.
X_vif_scaled = (
    X_vif - X_vif.mean()
) / X_vif.std(ddof=1)

# ------------------------------------------------------------
# Calculate VIF
# ------------------------------------------------------------

vif_records = []

for i, name in enumerate(channel_names):

    vif_value = variance_inflation_factor(
        X_vif_scaled.values,
        i
    )

    vif_records.append({
        "channel": name,
        "VIF": vif_value
    })

vif_df = (
    pd.DataFrame(vif_records)
    .sort_values("VIF", ascending=False)
    .reset_index(drop=True)
)

print("Variance Inflation Factors:")
display(vif_df.round(3))

# ------------------------------------------------------------
# Add interpretation category
# ------------------------------------------------------------

def classify_vif(v):

    if v < 5:
        return "Low / acceptable"

    elif v < 10:
        return "Moderate"

    else:
        return "High"

vif_df["Interpretation"] = vif_df["VIF"].apply(classify_vif)

print("\nVIF interpretation:")
display(vif_df.round(3))

# ------------------------------------------------------------
# Save table
# ------------------------------------------------------------

vif_output = (
    TABLE_DIR /
    "Paper1_14ch_VIF.csv"
)

vif_df.to_csv(
    vif_output,
    index=False
)

print("VIF table saved to:")
print(vif_output)

print("\nMaximum VIF:")
print(
    vif_df.loc[0, "channel"],
    "=",
    round(vif_df.loc[0, "VIF"], 3)
)

print("\nCell 11 completed.")

In [ ]:
# Recreate output directory variables

QC_DIR = PROJECT_ROOT / "03_outputs" / "qc"
TABLE_DIR = PROJECT_ROOT / "03_outputs" / "tables"
FIGURE_DIR = PROJECT_ROOT / "03_outputs" / "figures"

for folder in [QC_DIR, TABLE_DIR, FIGURE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("QC_DIR     :", QC_DIR)
print("TABLE_DIR  :", TABLE_DIR)
print("FIGURE_DIR :", FIGURE_DIR)

In [ ]:
# ============================================================
# Cell 12: Save final QC package and summary tables
# ============================================================

# ------------------------------------------------------------
# 1. Save geometry QC table
# ------------------------------------------------------------

geometry_qc_output = (
    QC_DIR /
    "Paper1_14ch_geometry_QC.csv"
)

geometry_qc.to_csv(
    geometry_qc_output,
    index=False
)

# ------------------------------------------------------------
# 2. Save missing-data summary
# ------------------------------------------------------------

missing_output = (
    QC_DIR /
    "Paper1_14ch_missing_data_QC.csv"
)

missing_df.to_csv(
    missing_output,
    index=False
)

# ------------------------------------------------------------
# 3. Create compact redundancy summary
# ------------------------------------------------------------

redundancy_rows = []

for i in range(len(channel_names)):
    for j in range(i + 1, len(channel_names)):

        redundancy_rows.append({
            "Variable_1": channel_names[i],
            "Variable_2": channel_names[j],
            "Pearson_r": pearson_corr.iloc[i, j],
            "Spearman_rho": spearman_corr.iloc[i, j],
            "Abs_Pearson_r": abs(pearson_corr.iloc[i, j]),
            "Abs_Spearman_rho": abs(spearman_corr.iloc[i, j])
        })

redundancy_df = pd.DataFrame(redundancy_rows)

# Sort by strongest Pearson relationship
redundancy_df = (
    redundancy_df
    .sort_values(
        "Abs_Pearson_r",
        ascending=False
    )
    .reset_index(drop=True)
)

redundancy_output = (
    TABLE_DIR /
    "Paper1_14ch_redundancy_summary.csv"
)

redundancy_df.to_csv(
    redundancy_output,
    index=False
)

# ------------------------------------------------------------
# 4. Create final channel decision table
# ------------------------------------------------------------

family_map = {
    "RTE_TMI": "Magnetic",
    "CBG": "Gravity",
    "CBG_RES": "Gravity",
    "HGM": "Gravity",
    "K": "Radiometric",
    "eTh": "Radiometric",
    "eU": "Radiometric",
    "CPD": "Thermal",
    "MAG_LD": "Structural",
    "GRAV_LD": "Structural",
    "DEM_LD": "Structural",
    "ID": "Structural",
    "DEM": "Terrain",
    "SLOPE": "Terrain",
}

role_map = {
    "RTE_TMI": "Magnetic field / susceptibility contrast",
    "CBG": "Broad density architecture",
    "CBG_RES": "Residual gravity variation",
    "HGM": "Lateral density-gradient boundaries",
    "K": "Potassium concentration",
    "eTh": "Equivalent thorium concentration",
    "eU": "Equivalent uranium concentration",
    "CPD": "Regional thermal structure",
    "MAG_LD": "Magnetic lineament density",
    "GRAV_LD": "Gravity lineament density",
    "DEM_LD": "Topographic lineament density",
    "ID": "Cross-network intersection density",
    "DEM": "Elevation / topographic setting",
    "SLOPE": "Terrain steepness",
}

decision_records = []

for name in channel_names:

    vif_value = vif_df.loc[
        vif_df["channel"] == name,
        "VIF"
    ].iloc[0]

    decision_records.append({
        "channel": name,
        "family": family_map[name],
        "physical_role": role_map[name],
        "VIF": vif_value,
        "primary_stack_status": "KEEP",
        "QC_note": (
            "Retain; evaluate redundancy with ID by ablation"
            if name == "MAG_LD"
            else
            "Retain; evaluate redundancy with MAG_LD by ablation"
            if name == "ID"
            else
            "Retain"
        )
    })

final_inventory_df = pd.DataFrame(decision_records)

inventory_output = (
    TABLE_DIR /
    "Paper1_FINAL_14ch_inventory.csv"
)

final_inventory_df.to_csv(
    inventory_output,
    index=False
)

# ------------------------------------------------------------
# 5. Save concise text summary
# ------------------------------------------------------------

summary_txt = (
    QC_DIR /
    "Paper1_Notebook01_QC_summary.txt"
)

with open(summary_txt, "w", encoding="utf-8") as f:

    f.write("PAPER 1 — NOTEBOOK 01 DATA CUBE AND QC SUMMARY\n")
    f.write("=" * 60 + "\n\n")

    f.write(f"Frozen channels: {len(channel_names)}\n")
    f.write(f"Master grid shape: {cube.shape[0]} rows x {cube.shape[1]} columns\n")
    f.write(f"Master-grid cells: {total_cells}\n")
    f.write(f"Common valid cells: {valid_cells}\n")
    f.write(f"Common valid coverage: {valid_pct:.2f}%\n\n")

    f.write("GEOMETRY QC\n")
    f.write("-" * 60 + "\n")
    f.write(
        f"Aligned rasters: "
        f"{int(geometry_qc['PASS'].sum())}/{len(geometry_qc)}\n\n"
    )

    f.write("STRONGEST PEARSON RELATIONSHIPS\n")
    f.write("-" * 60 + "\n")

    for _, row in redundancy_df.head(10).iterrows():

        f.write(
            f"{row['Variable_1']} - {row['Variable_2']}: "
            f"r = {row['Pearson_r']:.3f}, "
            f"rho = {row['Spearman_rho']:.3f}\n"
        )

    f.write("\nVIF RESULTS\n")
    f.write("-" * 60 + "\n")

    for _, row in vif_df.iterrows():

        f.write(
            f"{row['channel']}: "
            f"VIF = {row['VIF']:.3f}\n"
        )

    f.write("\nPRIMARY STACK DECISION\n")
    f.write("-" * 60 + "\n")
    f.write(
        "All 14 channels retained in the primary stack.\n"
    )
    f.write(
        "MAG_LD and ID are retained despite strong correlation because "
        "they represent different structural attributes. Their influence "
        "will be evaluated using ablation experiments.\n"
    )

# ------------------------------------------------------------
# 6. Print saved outputs
# ------------------------------------------------------------

print("Notebook 01 QC outputs saved successfully.\n")

print("Geometry QC:")
print(geometry_qc_output)

print("\nMissing-data QC:")
print(missing_output)

print("\nRedundancy summary:")
print(redundancy_output)

print("\nFinal 14-channel inventory:")
print(inventory_output)

print("\nQC text summary:")
print(summary_txt)

print("\nTop 10 redundancy relationships:")
display(
    redundancy_df[
        [
            "Variable_1",
            "Variable_2",
            "Pearson_r",
            "Spearman_rho"
        ]
    ]
    .head(10)
    .round(3)
)

print("\nFinal frozen inventory:")
display(
    final_inventory_df.round(3)
)

print("\nCell 12 completed.")